# 05 — BAR-ATH: label-free plausibility and (only if adopted) the candidate

Aggregates only. Full-data SetSeq fits (all 7 proxy matches, `target_aug` recipe, seeds 0-2) are decoded with the final configs selected on the `target_like` out-of-fold predictions. The plausibility numbers are reported, never used for a choice. The candidate CSV is written only if `results/decision.json` adopts it, and must rebuild byte-identically from an independent fit. No public-window check.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp2_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_2')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
import exp2_common as C
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp2_protocol as P, exp2_target as E
import exp1_common as C1
assert C.public_check_spent(), 'exp_1 public check marker missing'

PREREG verified 154a10f7cda6821e | frozen 2026-11-08T13:06:55+00:00


In [2]:
t0 = time.time()
decision = C1.read_json(C.RESULTS / 'decision.json')
match, item = E.load_target()
models = {arm: E.full_model(arm) for arm in P.ENSEMBLE}
plausibility = E.plausibility(match, item, models)
C1.write_json(C.RESULTS / 'target_plausibility.json', plausibility)
keys = ('dead_share', 'possessions_per_min', 'mean_possession_s', 'dead_share_minus_prior', 'possession_count')
print(pd.DataFrame({k: {m: v[m] for m in keys} for k, v in plausibility.items() if isinstance(v, dict)}).T.round(3).to_string())
print('published dead-share prior', plausibility['published_dead_share_prior'], '| ball coverage', round(plausibility['ball_coverage_graded'], 3))
print(f'{time.time() - t0:.0f}s')

                     dead_share  possessions_per_min  mean_possession_s  dead_share_minus_prior  possession_count
exp1_setseq_shipped       0.384                4.126              8.957                  -0.023             437.0
exp2_aug                  0.408                3.569              9.959                   0.001             378.0
exp2_aug_hsmm             0.409                3.692              9.602                   0.002             391.0
exp2_aug_ens3             0.408                3.749              9.481                   0.001             397.0
published dead-share prior 0.407 | ball coverage 0.593
186s


In [3]:
if decision['adopt_exp2_candidate']:
    summary = E.build_candidate(match, item, models['exp2_aug'])
    summary['reproduction'] = E.reproduce(match, item)
else:
    summary = {'adopted': False, 'reason': 'PREREG adoption rule (H1) not met; exp_1 candidate stays',
               'rules': {k: v['passed'] for k, v in decision['rules'].items()}}
C1.write_json(C.RESULTS / 'target_candidate.json', summary)
print(json.dumps({k: v for k, v in summary.items() if k != 'share_graded'}, indent=1))

{
 "sha256": "a59128e642d94e2dee60398498ba7c6a5304fcb068cbe223fce38128aee80aed",
 "format_valid": true,
 "decoder": {
  "temperature": 1.2822955751739529,
  "lambda_team": 1.0,
  "lambda_dead": 4.0,
  "gamma": 0.0
 },
 "public_labels_used": false,
 "graded_frames": 158857,
 "possession_count": 378,
 "abstain_frames_graded": 0,
 "dead_share": 0.4075426326822236,
 "possessions_per_min": 3.5692478140717756,
 "mean_possession_s": 9.95936507936508,
 "dead_share_minus_prior": 0.0005426326822236538,
 "reproduction": {
  "first_sha256": "a59128e642d94e2dee60398498ba7c6a5304fcb068cbe223fce38128aee80aed",
  "second_sha256": "a59128e642d94e2dee60398498ba7c6a5304fcb068cbe223fce38128aee80aed",
  "byte_identical": true
 }
}
